# Create a visual

In [34]:
import pandas as pd
import json

In [35]:
data = pd.read_excel('interim/cluster_review.xlsx')

In [36]:
# How many unique clusters after manual merging?
# This includes 'Noise'
data.name.nunique()

90

In [37]:
data[data.name != 'Noise'].name.nunique()

89

In [38]:
data_size = data.n_samples.sum()
noise_size = data[data.name.eq('Noise')].n_samples.sum()

In [39]:
data_size

30879

In [40]:
# What share of geospatial turns is noise?
round(noise_size / data_size * 100, 1)

46.4

In [41]:
# How many groups/themes?
groups = data.group.dropna().unique()
groups.size

14

In [42]:
data.name.drop_duplicates().str.endswith('*').sum()

20

In [43]:
group_counts = data.groupby(['group']).n_samples.sum().reset_index()
group_counts.sort_values('n_samples')

,group,n_samples
4,Cultural*,182
1,Biographical*,374
0,Administrative*,530
5,Events*,692
7,Misc,719
11,Temporal*,753
8,POIs & Commercial*,846
2,Business & finance,980
9,Physical*,1150
12,Trip planning,1165


In [44]:
group_counts[group_counts.group.str.endswith('Country-specific')].n_samples.sum() / group_counts.n_samples.sum()

0.1746991594606035

In [45]:
data[data.group.eq('Country-specific')].shape

(20, 9)

In [46]:
group_counts[group_counts.group.str.endswith('*')].n_samples.sum() / group_counts.n_samples.sum()

0.4496583418999819

In [72]:
colours = [
    '#1f78b4', '#a6cee3', '#b2df8a',
    '#33a02c', '#e31a1c', '#fb9a99', 
    '#fdbf6f', '#ff7f00', '#cab2d6',
    '#cccccc', '#6a3d9a', '#b15928',
    '#8dd3c7', '#bc80bd', '#bebada'
]

In [73]:
group_colours = { g:k for g, k in zip(group_counts.sort_values('n_samples', ascending=False).group, colours[:len(groups)]) }
group_colours

{'Writing & ideation': '#1f78b4',
 'Country-specific': '#a6cee3',
 'Statistical*': '#b2df8a',
 'Historical*': '#33a02c',
 'Trip planning': '#e31a1c',
 'Physical*': '#fb9a99',
 'Business & finance': '#fdbf6f',
 'POIs & Commercial*': '#ff7f00',
 'Temporal*': '#cab2d6',
 'Misc': '#cccccc',
 'Events*': '#6a3d9a',
 'Administrative*': '#b15928',
 'Biographical*': '#8dd3c7',
 'Cultural*': '#bc80bd'}

In [74]:
leaf_counts = data.groupby(['group', 'name']).n_samples.sum().reset_index().assign(
    pc=lambda df_: (df_.n_samples / data_size *100).round(1)
)

leaf_counts

,group,name,n_samples,pc
0,Administrative*,Addresses and ZIP codes*,76,0.2
1,Administrative*,Capital cities,390,1.3
2,Administrative*,US states and places,64,0.2
3,Biographical*,Birthplaces & residences*,374,1.2
4,Business & finance,Food permits,87,0.3
...,...,...,...,...
84,Writing & ideation,Write a short story,305,1.0
85,Writing & ideation,Write a song,55,0.2
86,Writing & ideation,Write an analysis,158,0.5
87,Writing & ideation,Write an application / cover letter,106,0.3


In [75]:
leaf_counts[leaf_counts.group.eq('Writing & ideation')]

,group,name,n_samples,pc
74,Writing & ideation,Fictional world building,501,1.6
75,Writing & ideation,Image-generation prompts,206,0.7
76,Writing & ideation,Tell me a joke,50,0.2
77,Writing & ideation,Tell me about a place,158,0.5
78,Writing & ideation,What if / hypothetical history scenarios,69,0.2
79,Writing & ideation,Write a (fake) news article,67,0.2
80,Writing & ideation,Write a blog/social media post,140,0.5
81,Writing & ideation,Write a poem,120,0.4
82,Writing & ideation,Write a review (hotel/resort),67,0.2
83,Writing & ideation,Write a script (university/college focus),1285,4.2


In [76]:
root = [{"id": "root", "name": "Geospatial first turns"}]

group_level = [{
    "id": row['group'],
    "name": row['group'],
    "parent": "root",
    "size": int(row['n_samples']),
    'colour': group_colours[row['group']],
    #'colour': '#999' if row['group'][-1] == '*' or row['group'] == 'Country-specific' else group_colours[row['group']],
} for _, row in group_counts.sort_values('n_samples', ascending=False).iterrows() ]

category_level = [{
    "id": row['name'],
    "name": row['name'],
    "parent": row['group'],
    "size": int(row['n_samples']),
    'colour': group_colours[row['group']],
    #'colour': '#999', #if row['name'][-1] == '*' or row['group'] == 'Country-specific' else group_colours[row['group']],
} for _, row in leaf_counts.sort_values('n_samples', ascending=False).iterrows()]

example_level = [{
    "id": cat['id'] + 'example',
    "name": data[data.name == cat['id']].iloc[0].sample_query,
    "parent": cat['id'],
    "size": 1,
} for cat in category_level]

In [77]:
nodes = root + group_level + category_level + example_level

In [78]:
with open('visual/data.json', 'w') as f:
    f.write(json.dumps(nodes, indent=4))

In [79]:
len(nodes)

193

In [55]:
#for _, row in data.sample_queries.apply(lambda x: sorted(eval(x), key=len)).to_frame().iterrows():
#    print('*****')
#    for i in range(10):
#        print(row.sample_queries[i])